
# One-Rec — catalog expansion harvest

Makes the catalog **age-agnostic**: the 597K item2vec vocab ends at MPD-2017 and
misses deep cuts. This notebook harvests from Deezer's anonymous API:

- **H1** top tracks (100) of the top 40K MPD artists — post-2017 releases AND deep
  cuts of known artists (the Crybaby + Big Star cases)
- **H2** 1-hop related-artist expansion — brand-new artists MPD never saw (Tai Verdes)
- **H3** genre charts + the calibration ground-truth targets

Candidates are deduped against the existing catalog, priority-ranked, and embedded
(Discogs-EffNet → frozen PCA-256, the SAME space as `audio_emb.parquet`). Output feeds
the audio-ANN retrieval channel. CPU + internet, no GPU. Time-budgeted + checkpointed:
a second session attaches this one's output and resumes embedding.


In [ ]:
import asyncio, glob, json, os, re, sys, time, unicodedata
from pathlib import Path

import numpy as np
import pandas as pd

# Latest essentia-tensorflow ships cp314-only wheels; Kaggle is Python 3.12 —
# pin the last cp312 build (proven in the audio-embeddings run).
%pip install -q essentia-tensorflow==2.1b6.dev1389 aiohttp
!wget -q -nc https://essentia.upf.edu/models/feature-extractors/discogs-effnet/discogs-effnet-bs64-1.pb
print("model:", os.path.getsize("discogs-effnet-bs64-1.pb") / 1e6, "MB")
T_SESSION = time.time()


In [ ]:
CFG = dict(
    top_artists=40_000,       # Crybaby is MPD artist rank ~35.8K; 40K covers ~98% of playlist mass
    top_tracks_per_artist=100,
    related_hop_from=4_000,   # related-artist hop sources (top artists by playlist count)
    related_max_new=8_000,    # cap on brand-new (non-MPD) artists pulled in
    new_artist_top=50,
    deezer_rps=8,             # 20% headroom under the 10 req/s anonymous quota
    dur_tol_ms=7_000,         # kills most remaster/live/extended mismatches
    batch=500,
    ckpt_every=2_000,
    embed_workers=4,          # one single-threaded TF per vCPU
    harvest_part_every=1_000, # artists per harvest checkpoint part
    time_budget_h=10.8,       # whole session; embed loop stops early so export always runs
    seed=42,
)
WORK = Path("/kaggle/working")

def input_glob(pattern):
    return sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))

meta = pd.read_parquet(input_glob("track_meta.parquet")[0])
vocab_ids = set(json.load(open(input_glob("id_map.json")[0])))      # flat list of 597,682 ids
embedded_ids = set()
for p in input_glob("audio_emb.parquet") + input_glob("extension_audio_emb.parquet"):
    embedded_ids |= set(pd.read_parquet(p, columns=["track_id"])["track_id"])
resolved_dz = set()
for p in input_glob("preview_resolution.parquet"):
    r = pd.read_parquet(p)
    resolved_dz |= set(r.loc[r["source"] == "deezer", "source_id"].astype(str))
print(f"meta {len(meta):,} | vocab {len(vocab_ids):,} | embedded {len(embedded_ids):,} | "
      f"resolved deezer ids {len(resolved_dz):,}")

artists = (meta.groupby("artist", sort=False)
           .agg(plc=("playlist_count", "sum"), n_tracks=("track_id", "count"))
           .sort_values("plc", ascending=False))
top_artists = artists.head(CFG["top_artists"]).reset_index()
print(f"{len(artists):,} artists; harvesting top {len(top_artists):,} "
      f"(min playlist count {int(top_artists['plc'].min())})")


## Matching + rate-limit helpers (proven in the audio-embeddings run)

In [ ]:
import aiohttp

VERSION_RE = re.compile(r"\b(live|karaoke|cover|tribute|acoustic|remix|instrumental)\b", re.I)


def norm(s):
    s = unicodedata.normalize("NFKD", s or "").encode("ascii", "ignore").decode().casefold()
    return re.sub(r"[^a-z0-9 ]+", " ", s).strip()


def toks(s):
    return set(norm(s).split())


def clean_title(name):
    name = re.sub(r"\(.*?\)|\[.*?\]", " ", name or "")
    return re.sub(r"\s*-\s*.*(remaster|version|edit|deluxe).*$", "", name, flags=re.I).strip()


def artist_ok(want, got):
    if norm(want) == norm(got):
        return True
    a, b = toks(want), toks(got)
    return len(a & b) / max(len(a | b), 1) >= 0.6


class RateLimiter:
    def __init__(self, rps):
        self.min_int = 1.0 / rps
        self.next_t = 0.0
        self.lock = asyncio.Lock()

    async def acquire(self):
        async with self.lock:
            now = time.monotonic()
            wait = self.next_t - now
            self.next_t = max(now, self.next_t) + self.min_int
        if wait > 0:
            await asyncio.sleep(wait)


DEEZER = RateLimiter(CFG["deezer_rps"])


async def deezer_get(session, url, params=None, retries=5):
    for attempt in range(retries):
        await DEEZER.acquire()
        try:
            async with session.get(url, params=params, timeout=aiohttp.ClientTimeout(total=15)) as resp:
                data = await resp.json(content_type=None)
        except Exception:
            await asyncio.sleep(2 * (attempt + 1))
            continue
        # Quota errors come back as HTTP 200 with an error body.
        if isinstance(data, dict) and data.get("error", {}).get("code") == 4:
            await asyncio.sleep(5)
            continue
        return data
    return None


HDRS = {"User-Agent": "one-rec-research/1.0"}

mpd_artist_norms = {norm(a) for a in artists.index}

# (artist_norm, title_norm) -> (track_id, duration_ms); highest playlist count wins
meta_sorted = meta.sort_values("playlist_count", ascending=False)
exact_key, clean_key = {}, {}
for tid, nm, ar, dur in zip(meta_sorted["track_id"], meta_sorted["name"],
                            meta_sorted["artist"], meta_sorted["duration_ms"]):
    an = norm(ar)
    exact_key.setdefault((an, norm(nm)), (tid, dur))
    clean_key.setdefault((an, norm(clean_title(nm))), (tid, dur))
print(f"match keys: {len(exact_key):,} exact, {len(clean_key):,} cleaned")

# Ground truth from evaluation/calibration.json — the seed itself is included so
# it gets a DIRECT audio vector (today it only resolves via same-artist proxy).
CALIBRATION = [
    ("Rewind", "Goldspot"),
    ("When The Lights Go Out", "Crybaby"),
    ("fake prophet", "Tai Verdes"),
    ("The Ballad Of El Goodo", "Big Star"),
]


## H1 — top tracks of the top 40K MPD artists (2 requests per artist)

In [ ]:
state = {"artist_cursor": 0, "h2_done": False, "h3_done": False}
for sf in [WORK / "harvest_state.json"] + [Path(p) for p in input_glob("harvest_state.json")]:
    if sf.exists():
        state.update(json.loads(sf.read_text()))
        break

# Once the consolidated queue exists (this session or an attached one), harvesting
# is over — later sessions are pure embed sessions.
QUEUE_READY = (WORK / "embed_queue.parquet").exists() or bool(input_glob("embed_queue.parquet"))


async def one_artist(session, name):
    d = await deezer_get(session, "https://api.deezer.com/search/artist",
                         {"q": name, "limit": "5"})
    cands = (d or {}).get("data", [])
    best = next((c for c in cands if norm(c.get("name", "")) == norm(name)), None)
    if best is None:
        ok = [c for c in cands if artist_ok(name, c.get("name", ""))]
        best = max(ok, key=lambda c: c.get("nb_fan", 0)) if ok else None
    if best is None:
        return name, None, []
    t = await deezer_get(session, f"https://api.deezer.com/artist/{best['id']}/top",
                         {"limit": str(CFG["top_tracks_per_artist"])})
    rows = [dict(deezer_id=int(tr["id"]), title=tr.get("title", ""),
                 artist_deezer=(tr.get("artist") or {}).get("name", ""),
                 artist_mpd=name, artist_dzid=int(best["id"]),
                 duration_s=int(tr.get("duration") or 0), pos=pos,
                 stage="artist")
            for pos, tr in enumerate((t or {}).get("data", []))]
    return name, int(best["id"]), rows


if not QUEUE_READY and state["artist_cursor"] < len(top_artists):
    t0 = time.time()
    while state["artist_cursor"] < len(top_artists):
        lo = state["artist_cursor"]
        hi = min(lo + CFG["harvest_part_every"], len(top_artists))
        chunk = top_artists.iloc[lo:hi]
        async with aiohttp.ClientSession(headers=HDRS) as session:
            out = await asyncio.gather(*(one_artist(session, n) for n in chunk["artist"]))
        rows = [r for _, _, rs in out for r in rs]
        n_miss = sum(1 for _, dzid, _ in out if dzid is None)
        pd.DataFrame(rows).to_parquet(WORK / f"harvest_artist_{lo:06d}.parquet", index=False)
        pd.DataFrame([{"artist_mpd": n, "artist_dzid": dzid} for n, dzid, _ in out
                      if dzid is not None]).to_parquet(WORK / f"artist_ids_{lo:06d}.parquet", index=False)
        state["artist_cursor"] = hi
        (WORK / "harvest_state.json").write_text(json.dumps(state))
        el_m = (time.time() - t0) / 60
        print(f"H1 {hi:,}/{len(top_artists):,} artists | +{len(rows):,} tracks | "
              f"{n_miss} misses | {el_m:.0f}m", flush=True)
        # Harvest never eats the whole session: past ~55% of budget, go embed
        # what we have; the cursor lets the next session finish H1.
        if (time.time() - T_SESSION) / 3600 > CFG["time_budget_h"] * 0.55:
            print("H1: time guard hit — yielding to the embed stage")
            break
print(f"H1 cursor: {state['artist_cursor']:,}/{len(top_artists):,}")


## H2 — related-artist hop (brand-new artists MPD never saw)

In [ ]:
H1_DONE = state["artist_cursor"] >= len(top_artists)

if not QUEUE_READY and H1_DONE and not state["h2_done"]:
    id_parts = sorted(set(glob.glob(str(WORK / "artist_ids_*.parquet")) +
                          input_glob("artist_ids_*.parquet")))
    aid = (pd.concat([pd.read_parquet(p) for p in id_parts])
           .drop_duplicates("artist_mpd")
           .merge(top_artists[["artist", "plc"]], left_on="artist_mpd", right_on="artist"))
    src = aid.nlargest(CFG["related_hop_from"], "plc")
    async with aiohttp.ClientSession(headers=HDRS) as session:
        rel = await asyncio.gather(*(deezer_get(session, f"https://api.deezer.com/artist/{i}/related",
                                                {"limit": "20"}) for i in src["artist_dzid"]))
        new_artists = {}
        for d in rel:
            for c in (d or {}).get("data", []):
                if norm(c.get("name", "")) not in mpd_artist_norms and c["id"] not in new_artists:
                    new_artists[c["id"]] = (c.get("name", ""), c.get("nb_fan") or 0)
        picked = sorted(new_artists.items(), key=lambda kv: -kv[1][1])[:CFG["related_max_new"]]
        print(f"H2: {len(new_artists):,} unknown artists found, taking top {len(picked):,} by fans")
        tops = await asyncio.gather(*(deezer_get(session, f"https://api.deezer.com/artist/{i}/top",
                                                 {"limit": str(CFG["new_artist_top"])})
                                      for i, _ in picked))
    rows = []
    for (dzid, (nm, fans)), t in zip(picked, tops):
        rows.extend(dict(deezer_id=int(tr["id"]), title=tr.get("title", ""),
                         artist_deezer=(tr.get("artist") or {}).get("name", ""),
                         artist_mpd="", artist_dzid=dzid,
                         duration_s=int(tr.get("duration") or 0), pos=pos, stage="related")
                    for pos, tr in enumerate((t or {}).get("data", [])))
    pd.DataFrame(rows).to_parquet(WORK / "harvest_related.parquet", index=False)
    state["h2_done"] = True
    (WORK / "harvest_state.json").write_text(json.dumps(state))
    print(f"H2: {len(rows):,} tracks from related artists")


## H3 — genre charts + calibration targets

In [ ]:
if not QUEUE_READY and H1_DONE and not state["h3_done"]:
    rows = []
    async with aiohttp.ClientSession(headers=HDRS) as session:
        g = await deezer_get(session, "https://api.deezer.com/genre")
        gids = [x["id"] for x in (g or {}).get("data", [])]
        charts = await asyncio.gather(*(deezer_get(session, f"https://api.deezer.com/chart/{gid}/tracks",
                                                   {"limit": "100"}) for gid in gids))
        for d in charts:
            rows.extend(dict(deezer_id=int(tr["id"]), title=tr.get("title", ""),
                             artist_deezer=(tr.get("artist") or {}).get("name", ""),
                             artist_mpd="", artist_dzid=int((tr.get("artist") or {}).get("id") or 0),
                             duration_s=int(tr.get("duration") or 0), pos=pos, stage="chart")
                        for pos, tr in enumerate((d or {}).get("data", [])))
        for title, artist in CALIBRATION:
            d = await deezer_get(session, "https://api.deezer.com/search",
                                 {"q": f"{artist} {title}", "limit": "5"})
            for tr in (d or {}).get("data", []):
                if (artist_ok(artist, (tr.get("artist") or {}).get("name", ""))
                        and norm(title) in norm(tr.get("title", ""))):
                    rows.append(dict(deezer_id=int(tr["id"]), title=tr.get("title", ""),
                                     artist_deezer=(tr.get("artist") or {}).get("name", ""),
                                     artist_mpd="", artist_dzid=int((tr.get("artist") or {}).get("id") or 0),
                                     duration_s=int(tr.get("duration") or 0), pos=0,
                                     stage="calibration"))
                    break
    pd.DataFrame(rows).to_parquet(WORK / "harvest_extra.parquet", index=False)
    state["h3_done"] = True
    (WORK / "harvest_state.json").write_text(json.dumps(state))
    print(f"H3: {len(rows):,} chart/calibration tracks "
          f"({sum(r['stage'] == 'calibration' for r in rows)}/4 calibration hits)")


## Consolidate — dedup against the existing catalog, priority-rank

`kind`: **new** (not in MPD at all, gets a `dz:` id) / **mpd_ext** (in MPD metadata but
NOT in the 597K item2vec vocab — currently just as unreachable as new tracks, keeps its
Spotify id) / **backfill** (in-vocab track gaining an audio vector, widening the future
audio-ANN channel).

In [ ]:
def build_queue():
    part_files = sorted(set(glob.glob(str(WORK / "harvest_artist_*.parquet")) +
                            input_glob("harvest_artist_*.parquet") +
                            glob.glob(str(WORK / "harvest_related.parquet")) +
                            input_glob("harvest_related.parquet") +
                            glob.glob(str(WORK / "harvest_extra.parquet")) +
                            input_glob("harvest_extra.parquet")))
    frames = [f for f in (pd.read_parquet(p) for p in part_files) if len(f)]
    rows = pd.concat(frames, ignore_index=True)
    # calibration rows first so drop_duplicates keeps their stage label
    rows["is_cal"] = (rows["stage"] == "calibration").astype(int)
    rows = (rows.sort_values("is_cal", ascending=False)
            .drop_duplicates("deezer_id").reset_index(drop=True))

    # Match to the MPD catalog: known-artist rows match under their MPD artist name,
    # chart/related rows under the Deezer artist name.
    a_norm = np.where(rows["artist_mpd"] != "", rows["artist_mpd"].map(norm),
                      rows["artist_deezer"].map(norm))
    t_norm = rows["title"].map(norm)
    c_norm = rows["title"].map(lambda s: norm(clean_title(s)))
    matched = []
    for a, t, c, dur_s in zip(a_norm, t_norm, c_norm, rows["duration_s"]):
        hit = exact_key.get((a, t)) or clean_key.get((a, c))
        if hit and dur_s and hit[1] and abs(hit[1] - dur_s * 1000) > CFG["dur_tol_ms"]:
            hit = None  # same name, different recording (live/extended) — treat as new
        matched.append(hit[0] if hit else "")
    rows["matched_track_id"] = matched
    rows["kind"] = np.where(rows["matched_track_id"] == "", "new",
                            np.where(rows["matched_track_id"].isin(vocab_ids),
                                     "backfill", "mpd_ext"))
    rows["id"] = np.where(rows["matched_track_id"] == "",
                          "dz:" + rows["deezer_id"].astype(str), rows["matched_track_id"])

    # Skip audio we already have (by catalog id or by previously-resolved Deezer id).
    rows = rows[~rows["id"].isin(embedded_ids)
                & ~rows["deezer_id"].astype(str).isin(resolved_dz)]
    rows = rows.drop_duplicates("id").reset_index(drop=True)

    # Priority: calibration first, then artist-weight x within-artist rank, with new
    # catalog entries ahead of backfill — the embed budget lands where it matters.
    plc_of = dict(zip(top_artists["artist"], top_artists["plc"]))
    max_plc = np.log1p(float(top_artists["plc"].max()))
    aw = rows["artist_mpd"].map(lambda a: np.log1p(plc_of.get(a, 0)) / max_plc if a else 0.35)
    rw = 1.0 / np.log2(2 + rows["pos"].astype(float))
    kw = rows["kind"].map({"new": 1.0, "mpd_ext": 0.9, "backfill": 0.5})
    rows["priority"] = kw * (0.65 * aw + 0.35 * rw)
    return (rows.sort_values(["is_cal", "priority"], ascending=[False, False])
            .reset_index(drop=True))


existing_q = ([WORK / "embed_queue.parquet"] +
              [Path(p) for p in input_glob("embed_queue.parquet")])
existing_q = [p for p in existing_q if p.exists()]
if existing_q:
    queue = pd.read_parquet(existing_q[0])
    print(f"queue loaded from {existing_q[0]}")
elif H1_DONE and state["h2_done"] and state["h3_done"]:
    queue = build_queue()
    queue.to_parquet(WORK / "embed_queue.parquet", index=False)
else:
    # Harvest incomplete (time guard) — embed this session from a PROVISIONAL queue;
    # the durable embed_queue.parquet is only written once harvesting finishes.
    queue = build_queue()
    print("provisional queue (harvest unfinished — not persisted)")
print(queue.groupby(["stage", "kind"]).size().to_string())
print(f"queue: {len(queue):,} tracks to embed")


## Embedder — Discogs-EffNet fork pool (one single-threaded TF per vCPU)

In [ ]:
import multiprocessing as mp
import tempfile

_model = None


def _init_worker():
    global _model, _MonoLoader
    os.environ["OMP_NUM_THREADS"] = "1"
    os.environ["TF_NUM_INTRAOP_THREADS"] = "1"
    os.environ["TF_NUM_INTEROP_THREADS"] = "1"
    from essentia.standard import MonoLoader, TensorflowPredictEffnetDiscogs
    _MonoLoader = MonoLoader
    _model = TensorflowPredictEffnetDiscogs(graphFilename="discogs-effnet-bs64-1.pb",
                                            output="PartitionedCall:1")


def _embed_one(args):
    tid, blob = args
    f = tempfile.NamedTemporaryFile(suffix=".mp3", delete=False)
    try:
        f.write(blob)
        f.close()
        audio = _MonoLoader(filename=f.name, sampleRate=16000, resampleQuality=4)()
        if len(audio) < 16000:  # <1s of audio — corrupt download
            return tid, None
        emb = _model(audio).mean(axis=0).astype(np.float32)
        return tid, emb
    except Exception:
        return tid, None
    finally:
        os.unlink(f.name)


pool = mp.get_context("fork").Pool(CFG["embed_workers"], initializer=_init_worker)


def embed_batch(payloads):
    return [(tid, emb) for tid, emb in pool.map(_embed_one, payloads) if emb is not None]


## Embed loop — fresh preview URL per track (they expire in ~15 min), overlapped
download/embed, npz checkpoints

In [ ]:
from concurrent.futures import ThreadPoolExecutor

done_ids, emb_chunks = set(), []
ckpt_files = sorted(WORK.glob("ext_ckpt_*.npz")) + [Path(p) for p in input_glob("ext_ckpt_*.npz")]
for f in ckpt_files:
    z = np.load(f, allow_pickle=True)
    emb_chunks.append((z["ids"], z["embs"]))
    done_ids.update(z["ids"].tolist())
miss_ids = set()
for p in [WORK / "ext_misses.parquet"] + [Path(q) for q in input_glob("ext_misses.parquet")]:
    if p.exists():
        miss_ids |= set(pd.read_parquet(p)["id"])

todo = queue[~queue["id"].isin(done_ids) & ~queue["id"].isin(miss_ids)].reset_index(drop=True)
print(f"embed: {len(done_ids):,} done, {len(miss_ids):,} known misses, {len(todo):,} queued")

DL_SEM = asyncio.Semaphore(12)


async def fetch_blob(session, dzid):
    # Harvested preview URLs are hours old — always re-fetch (TTL ~15 min).
    d = await deezer_get(session, f"https://api.deezer.com/track/{dzid}")
    url = (d or {}).get("preview") or ""
    if not url:
        return None
    async with DL_SEM:
        for _ in range(3):
            try:
                async with session.get(url, timeout=aiohttp.ClientTimeout(total=30)) as resp:
                    if resp.status == 200:
                        return await resp.read()
            except Exception:
                pass
            await asyncio.sleep(1)
    return None


async def _download_batch(rows):
    async with aiohttp.ClientSession(headers=HDRS) as session:
        blobs = await asyncio.gather(*(fetch_blob(session, d) for d in rows["deezer_id"]))
    return list(zip(rows["id"], blobs))


def download_batch(rows):
    return asyncio.run(_download_batch(rows))


pending_ids, pending_embs, new_misses = [], [], []


def flush(force=False):
    global pending_ids, pending_embs
    if pending_ids and (force or len(pending_ids) >= CFG["ckpt_every"]):
        n = len(list(WORK.glob("ext_ckpt_*.npz")))
        arr_ids = np.array(pending_ids, dtype=object)
        arr_embs = np.stack(pending_embs).astype(np.float16)
        np.savez(WORK / f"ext_ckpt_{n:03d}.npz", ids=arr_ids, embs=arr_embs)
        emb_chunks.append((arr_ids, arr_embs))
        pending_ids, pending_embs = [], []
    if new_misses:
        pd.DataFrame({"id": sorted(miss_ids | set(new_misses))}).to_parquet(
            WORK / "ext_misses.parquet", index=False)


batches = [todo.iloc[i:i + CFG["batch"]] for i in range(0, len(todo), CFG["batch"])]
throughput_printed = False
with ThreadPoolExecutor(max_workers=1) as fetcher:
    future = fetcher.submit(download_batch, batches[0]) if batches else None
    for bi in range(len(batches)):
        pairs = future.result()
        if bi + 1 < len(batches):
            future = fetcher.submit(download_batch, batches[bi + 1])
        payloads = [(i, b) for i, b in pairs if b]
        new_misses.extend(i for i, b in pairs if not b)
        t0 = time.time()
        for tid, emb in embed_batch(payloads):
            pending_ids.append(tid)
            pending_embs.append(emb)
        if not throughput_printed and payloads:
            rate = len(payloads) / max(time.time() - t0, 1e-9)
            print(f"embed throughput: {rate:.2f} clips/s "
                  f"({'OK' if rate >= 1.5 else 'SLOW — checkpoints cover extra sessions'})")
            throughput_printed = True
        flush()
        if bi % 10 == 0:
            total = sum(len(c[0]) for c in emb_chunks) + len(pending_ids)
            print(f"batch {bi + 1}/{len(batches)} | embedded {total:,} | "
                  f"misses {len(miss_ids) + len(new_misses):,} | "
                  f"{(time.time() - T_SESSION) / 3600:.1f}h", flush=True)
        if (time.time() - T_SESSION) / 3600 > CFG["time_budget_h"]:
            print(f"time budget reached at batch {bi + 1} — stopping cleanly")
            break
flush(force=True)
print("embed loop done")


## Export — frozen-PCA projection (SAME space as audio_emb.parquet) +
extension catalog metadata

In [ ]:
all_ids = (np.concatenate([i for i, _ in emb_chunks])
           if emb_chunks else np.array([], dtype=object))
all_embs = (np.concatenate([e.astype(np.float32) for _, e in emb_chunks])
            if emb_chunks else np.zeros((0, 1280), np.float32))
_, keep = np.unique(all_ids, return_index=True)
all_ids, all_embs = all_ids[np.sort(keep)], all_embs[np.sort(keep)]

# The FROZEN transform from the first audio run — extension vectors must live in
# the same 256-dim space as audio_emb.parquet or cosine features/ANN break.
pca = np.load(input_glob("pca_audio.npz")[0])
reduced = ((all_embs - pca["mean"]) @ pca["components"].T).astype(np.float16)

pd.DataFrame({"track_id": all_ids, "embedding": [r.tobytes() for r in reduced]}
             ).to_parquet(WORK / "extension_audio_emb.parquet", index=False)

got = set(all_ids.tolist())
ext = queue[queue["kind"] != "backfill"].copy()
ext["embedded"] = ext["id"].isin(got)
ext["duration_ms"] = ext["duration_s"] * 1000
ext[["id", "deezer_id", "title", "artist_deezer", "artist_mpd", "duration_ms",
     "pos", "stage", "kind", "matched_track_id", "embedded"]].to_parquet(
    WORK / "extension_tracks.parquet", index=False)

print(f"embedded total: {len(all_ids):,} "
      f"({int((queue['kind'] != 'backfill').sum()):,} extension candidates in queue)")
print(queue[queue["id"].isin(got)].groupby(["stage", "kind"]).size().to_string())

print("\ncalibration targets:")
for r in queue[queue["stage"] == "calibration"].itertuples():
    print(f"  {r.title} — {r.artist_deezer}: id={r.id} kind={r.kind} "
          f"embedded={'YES' if r.id in got else 'no'}")

remaining = len(queue) - len(queue[queue["id"].isin(got)]) - len(miss_ids) - len(set(new_misses))
print(f"\nremaining queue for a follow-up session: ~{max(remaining, 0):,}")
print("files:", sorted(f.name for f in WORK.glob("*.parquet")))
